# Machine Learning Models Implementation

This module implements elastic net regression and boosting(LightGBM) models. Compared to linear regression. Return average MSE from cross validation.
Since the ytm data has many outliers, we also use classification models to predict the direction of movement.

In [1]:
import pandas as pd

data = pd.read_parquet('../data/final_merged_clean.parquet', engine='pyarrow')

data.sort_values(by=['bond_cusip', 'date'], inplace=True)

In [2]:
macro_columns = ["sp500_ret", "ir3m_chg", "ir10y_chg",
    "vix_chg", "gdp_gr", "cpi_infl"] # Macro feature columns which need normalization
data = data.drop(columns=['gs3m', 'term_spread']) # Duplicates of ir3m and ir10y

data.loc[:, 'ytm_chg'] = data.groupby(['bond_cusip'])['ytm'].diff()

data.dropna(subset=['ytm_chg'], inplace=True)

data['up_down'] = data['ytm_chg'].apply(lambda x: 'up' if x > 1e-6 else ('down' if x < -1e-6 else 'neutral'))


In [3]:
# Set parameters
train_year = 10
val_year = 1
test_year = 1
random_seed = 666

start_year = data['date'].min().year
end_year = data['date'].max().year - test_year - val_year - train_year

# Boosting hyperparameters
max_leaves = [15, 31, 63]
num_tree = 500
learning_speed = [0.05, 0.1]

# Elastic Net hyperparameters
l1_ratios = [0.2, 0.5, 0.8]
alphas = [0.05, 0.5, 5]

In [4]:
idx_cols = ['bond_cusip', 'date']
y_col_reg = 'ytm_chg'
y_col_clf = 'up_down'
x_cols = [col for col in data.columns if col not in idx_cols + [y_col_reg, y_col_clf, 'ytm']]  # Exclude target and index columns from features


In [5]:
from sklearn.linear_model import ElasticNetCV, ElasticNet
from sklearn.linear_model import LinearRegression, LogisticRegressionCV
from sklearn.linear_model import LogisticRegression
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, median_absolute_error, accuracy_score, precision_score, recall_score

from sklearn.preprocessing import StandardScaler

from lightgbm import LGBMRegressor, LGBMClassifier, early_stopping


In [6]:
def normalize(train_data: pd.DataFrame,
              test_data: pd.DataFrame,
              cols: list) -> pd.DataFrame:
    scaler = StandardScaler()
    scaler.fit(train_data[cols])
    train_scaled= scaler.transform(train_data[cols])
    test_scaled = scaler.transform(test_data[cols])
    train_data.loc[:, cols] = train_scaled
    test_data.loc[:, cols] = test_scaled
    return train_data, test_data

In [7]:

def calc_metrics(y_pred, y_test, task_type: str):
    if task_type == 'regression':
        mse = mean_squared_error(y_test, y_pred)
        mae = mean_absolute_error(y_test, y_pred)
        medae = median_absolute_error(y_test, y_pred)
        r2 = r2_score(y_test, y_pred)
        return mse, mae, medae, r2
    elif task_type == 'classification':
        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred, average='weighted', zero_division=0)
        recall = recall_score(y_test, y_pred, average='weighted', zero_division=0)
        return accuracy, precision, recall
    else:
        raise ValueError("Unsupported task type")

In [8]:
def data_split(data: pd.DataFrame,
             x_cols: list,
             y_col: str,
             index_cols: list,
             year_: int,
             train_year: int,
             val_year: int,
             test_year: int,
             ) -> pd.DataFrame:
    

    data_train = data[(data['date'].dt.year <= (year_ + train_year)) & 
                      (data['date'].dt.year >= year_)]
    data_val = data[(data['date'].dt.year > (year_ + train_year)) & 
                    (data['date'].dt.year <= (year_ + train_year + val_year))]

    data_test = data[(data['date'].dt.year > (year_ + train_year + val_year)) & 
                     (data['date'].dt.year <= (year_ + train_year + val_year + test_year))]

    data_train, data_test = normalize(data_train, data_test, macro_columns)
    data_train, data_val = normalize(data_train, data_val, macro_columns)
    X_train = data_train[x_cols]
    y_train = data_train[y_col]
    X_test = data_test[x_cols]
    y_test = data_test[y_col]
    X_val = data_val[x_cols]
    y_val = data_val[y_col]
    idx_train = data_train[index_cols]
    idx_val = data_val[index_cols]
    idx_test = data_test[index_cols]

    return X_train, y_train, idx_train, X_val, y_val, idx_val, X_test, y_test, idx_test

## Linear Regression

In [9]:
# linear regression
lm = LinearRegression()

df_metrics_lm = pd.DataFrame(columns=['start_date', 'end_date', 'model'])
df_results_lm = pd.DataFrame(columns=['bond_cusip', 'date', 'model', 'pred', 'act'])

for year_ in range(start_year, end_year + 1):
    print(f"Forecasting for period starting {year_ + train_year + val_year} to {year_ + train_year + val_year + test_year}")
    X_train, y_train, idx_train, X_val, y_val, idx_val, X_test, y_test, idx_test = data_split(data, x_cols, y_col_reg, idx_cols,
                                                                                     year_, train_year, val_year, test_year)
    X_train_full = pd.concat([X_train, X_val], ignore_index=True)
    y_train_full = pd.concat([y_train, y_val], ignore_index=True)
    lm.fit(X_train_full, y_train_full)
    y_pred = lm.predict(X_test)
    mse, mae, medae, r2 = calc_metrics(y_pred, y_test, task_type='regression')
    df_metrics_lm = pd.concat([df_metrics_lm, pd.DataFrame({'start_year': [year_ + train_year + val_year],
                                                            'end_year': [year_ + train_year + val_year + test_year],
                                                            'model': ['LinearRegression'],
                                                            'mse': [mse],
                                                            'mae': [mae],
                                                            'medae': [medae],
                                                            'r2': [r2]})], ignore_index=True)
    df_results_lm = pd.concat([df_results_lm, pd.DataFrame({'bond_cusip': idx_test['bond_cusip'],
                                                            'year': idx_test['date'],
                                                            'model': ['LinearRegression'] * len(y_test),
                                                            'pred': y_pred,
                                                            'act': y_test})], ignore_index=True)



Forecasting for period starting 2013 to 2014


/var/folders/hw/dfkspcjj1bv53cx3_42lwrtc0000gn/T/ipykernel_40006/424365052.py:23: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_results_lm = pd.concat([df_results_lm, pd.DataFrame({'bond_cusip': idx_test['bond_cusip'],


Forecasting for period starting 2014 to 2015
Forecasting for period starting 2015 to 2016
Forecasting for period starting 2016 to 2017
Forecasting for period starting 2017 to 2018
Forecasting for period starting 2018 to 2019
Forecasting for period starting 2019 to 2020
Forecasting for period starting 2020 to 2021
Forecasting for period starting 2021 to 2022
Forecasting for period starting 2022 to 2023
Forecasting for period starting 2023 to 2024


## Elastic Net Regression

In [10]:


df_metrics_en = pd.DataFrame(columns=['start_date', 'end_date', 'model'])
df_results_en = pd.DataFrame(columns=['bond_cusip', 'date', 'model', 'pred', 'act'])
hypoerparams_en = {}

for year_ in range(start_year, end_year + 1):
    print(f"Tuning and Forecasting for period starting {year_ + train_year + val_year} to {year_ + train_year + val_year + test_year}")
    X_train, y_train, idx_train, X_val, y_val, idx_val, X_test, y_test, idx_test = data_split(data, x_cols, y_col_reg, idx_cols,
                                                                                     year_, train_year, val_year, test_year)
    X_train_full = pd.concat([X_train, X_val], ignore_index=True)
    y_train_full = pd.concat([y_train, y_val], ignore_index=True)

    # Tuning Alpha and L1 ratio with cross-validation once and use for all years as tuning per year is computationally expensive
    elastic_net_cv = ElasticNetCV(l1_ratio=l1_ratios, 
                                alphas=alphas, 
                                cv=5,
                                random_state=random_seed,
                                n_jobs=-1)
    elastic_net_cv.fit(X_train_full, y_train_full)
    best_alpha = elastic_net_cv.alpha_
    best_l1_ratio = elastic_net_cv.l1_ratio_

    hypoerparams_en[year_] = {'alpha': best_alpha, 'l1_ratio': best_l1_ratio}

    elastic_net = ElasticNet(alpha=best_alpha, l1_ratio=best_l1_ratio, random_state=random_seed)
    elastic_net.fit(X_train_full, y_train_full)
    y_pred = elastic_net.predict(X_test)
    mse, mae, medae, r2 = calc_metrics(y_pred, y_test, task_type='regression')
    df = pd.DataFrame({'start_year': [year_ + train_year + val_year],
                       'end_year': [year_ + train_year + val_year + test_year],
                       'model': ['ElasticNet'],
                       'mse': [mse],
                       'mae': [mae],
                       'medae': [medae],
                       'r2': [r2]})
    df_result = pd.DataFrame({'bond_cusip': idx_test['bond_cusip'],
                              'year': idx_test['date'],
                              'model': ['ElasticNet'] * len(y_test),
                              'pred': y_pred,
                              'act': y_test})   

    df_metrics_en = pd.concat([df_metrics_en, df], ignore_index=True)
    df_results_en = pd.concat([df_results_en, df_result], ignore_index=True)


Tuning and Forecasting for period starting 2013 to 2014


/var/folders/hw/dfkspcjj1bv53cx3_42lwrtc0000gn/T/ipykernel_40006/2350610850.py:42: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_results_en = pd.concat([df_results_en, df_result], ignore_index=True)


Tuning and Forecasting for period starting 2014 to 2015
Tuning and Forecasting for period starting 2015 to 2016
Tuning and Forecasting for period starting 2016 to 2017
Tuning and Forecasting for period starting 2017 to 2018
Tuning and Forecasting for period starting 2018 to 2019
Tuning and Forecasting for period starting 2019 to 2020
Tuning and Forecasting for period starting 2020 to 2021
Tuning and Forecasting for period starting 2021 to 2022
Tuning and Forecasting for period starting 2022 to 2023
Tuning and Forecasting for period starting 2023 to 2024


## Boosting Regression (LightGBM)

In [11]:


df_metrics_lgbm = pd.DataFrame(columns=['start_date', 'end_date', 'model'])
df_results_lgbm = pd.DataFrame(columns=['bond_cusip', 'date', 'model', 'pred', 'act'])
hypoerparams_lgmb = {}

callbacks = [early_stopping(stopping_rounds=50)]

for year_ in range(start_year, end_year + 1):
    print(f"Tuning and Forecasting for period starting {year_ + train_year + val_year} to {year_ + train_year + val_year + test_year}")
    X_train, y_train, idx_train, X_val, y_val, idx_val, X_test, y_test, idx_test = data_split(data, x_cols, y_col_reg, idx_cols,
                                                                                     year_, train_year, val_year, test_year)
    best_mse = float('inf')

    for leaves in max_leaves:
        for speed in learning_speed:
            print(f"Training LGBM with leaves={leaves}, trees={num_tree}, learning_rate={speed}")
            lgbm_reg = LGBMRegressor(max_leaves=leaves, n_estimators=num_tree, learning_rate=speed, random_state=random_seed, n_jobs=-1, verbose=-1)

            # Fit model with early stopping and validation set
            
            lgbm_reg.fit(X_train, y_train,
                         eval_set=[(X_val, y_val)],
                         callbacks=callbacks)
            y_val_pred = lgbm_reg.predict(X_val)
            mse_val, _, _, _ = calc_metrics(y_val_pred, y_val, task_type='regression')
            if mse_val < best_mse:
                best_mse = mse_val
                hypoerparams_lgmb[year_] = {'num_leaves': leaves, 'learning_rate': speed}
    
    # Predict with best params
    lgbm_reg_best = LGBMRegressor(max_leaves=hypoerparams_lgmb[year_]['num_leaves'], n_estimators=num_tree,
                                  learning_rate=hypoerparams_lgmb[year_]['learning_rate'], random_state=random_seed, n_jobs=-1, verbose=-1)
    lgbm_reg_best.fit(pd.concat([X_train, X_val], ignore_index=True),
                          pd.concat([y_train, y_val], ignore_index=True),
                          eval_set=[(X_val, y_val)],
                          callbacks=callbacks)
    y_pred = lgbm_reg_best.predict(X_test)
    mse, mae, medae, r2 = calc_metrics(y_pred, y_test, task_type='regression')
    df = pd.DataFrame({'start_year': [year_ + train_year + val_year],
                       'end_year': [year_ + train_year + val_year + test_year],
                       'model': ['LGBMRegressor'],
                       'mse': [mse],
                       'mae': [mae],
                       'medae': [medae],
                       'r2': [r2]})
    df_result = pd.DataFrame({'bond_cusip': idx_test['bond_cusip'],
                              'year': idx_test['date'],
                              'model': ['LGBMRegressor'] * len(y_test),
                              'pred': y_pred,
                              'act': y_test})  
    df_metrics_lgbm = pd.concat([df_metrics_lgbm, df], ignore_index=True)
    df_results_lgbm = pd.concat([df_results_lgbm, df_result], ignore_index=True)
            

Tuning and Forecasting for period starting 2013 to 2014
Training LGBM with leaves=15, trees=500, learning_rate=0.05
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[50]	valid_0's l2: 0.000155838
Training LGBM with leaves=15, trees=500, learning_rate=0.1
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[59]	valid_0's l2: 0.000156207
Training LGBM with leaves=31, trees=500, learning_rate=0.05
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[50]	valid_0's l2: 0.000155838
Training LGBM with leaves=31, trees=500, learning_rate=0.1
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[59]	valid_0's l2: 0.000156207
Training LGBM with leaves=63, trees=500, learning_rate=0.05
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[50]	valid_0's l2: 0.000155838
Training LG

/var/folders/hw/dfkspcjj1bv53cx3_42lwrtc0000gn/T/ipykernel_40006/367620823.py:51: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_results_lgbm = pd.concat([df_results_lgbm, df_result], ignore_index=True)


Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[83]	valid_0's l2: 7.0634e-05
Training LGBM with leaves=15, trees=500, learning_rate=0.1
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[42]	valid_0's l2: 7.10168e-05
Training LGBM with leaves=31, trees=500, learning_rate=0.05
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[83]	valid_0's l2: 7.0634e-05
Training LGBM with leaves=31, trees=500, learning_rate=0.1
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[42]	valid_0's l2: 7.10168e-05
Training LGBM with leaves=63, trees=500, learning_rate=0.05
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[83]	valid_0's l2: 7.0634e-05
Training LGBM with leaves=63, trees=500, learning_rate=0.1
Training until validation scores don't improve for 50 rounds
Early stop

## Ensemble

In [12]:
from sklearn.ensemble import StackingRegressor


df_metrics_stack = pd.DataFrame(columns=['start_date', 'end_date', 'model'])
df_results_stack = pd.DataFrame(columns=['bond_cusip', 'date', 'model', 'pred', 'act'])

for year_ in range(start_year, end_year + 1):
    print(f"Forecasting for period starting {year_ + train_year + val_year} to {year_ + train_year + val_year + test_year}")
    X_train, y_train, idx_train, X_val, y_val, idx_val, X_test, y_test, idx_test = data_split(data, x_cols, y_col_reg, idx_cols,
                                                                                     year_, train_year, val_year, test_year)
    X_train_full = pd.concat([X_train, X_val], ignore_index=True)
    y_train_full = pd.concat([y_train, y_val], ignore_index=True)

     # Use previously determined best hyperparameters

    elastic_net = ElasticNet(alpha=hypoerparams_en[year_]['alpha'], l1_ratio=hypoerparams_en[year_]['l1_ratio'], random_state=random_seed)
    lgbm = LGBMRegressor(max_leaves=hypoerparams_lgmb[year_]['num_leaves'],
                        n_estimators=num_tree,
                        learning_rate=hypoerparams_lgmb[year_]['learning_rate'],
                        random_state=random_seed, n_jobs=-1, verbose=-1)
    base_models = [
        ('elastic_net', elastic_net),
        ('lgbm', lgbm)
    ]
    meta_model = LinearRegression()
    stacked_model = StackingRegressor(estimators=base_models, final_estimator=meta_model, n_jobs=-1)
    stacked_model.fit(X_train_full, y_train_full)
    y_pred = stacked_model.predict(X_test)
    mse, mae, medae, r2 = calc_metrics(y_pred, y_test, task_type='regression')
    df = pd.DataFrame({'start_year': [year_ + train_year + val_year],
                       'end_year': [year_ + train_year + val_year + test_year],
                       'model': ['StackingRegressor'],
                       'mse': [mse],
                       'mae': [mae],
                       'medae': [medae],
                       'r2': [r2]})         
    df_result = pd.DataFrame({'bond_cusip': idx_test['bond_cusip'],
                              'year': idx_test['date'],
                              'model': ['StackingRegressor'] * len(y_test),
                              'pred': y_pred,
                              'act': y_test})  
    df_metrics_stack = pd.concat([df_metrics_stack, df], ignore_index=True)
    df_results_stack = pd.concat([df_results_stack, df_result], ignore_index=True)

Forecasting for period starting 2013 to 2014


/var/folders/hw/dfkspcjj1bv53cx3_42lwrtc0000gn/T/ipykernel_40006/2781574647.py:43: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_results_stack = pd.concat([df_results_stack, df_result], ignore_index=True)


Forecasting for period starting 2014 to 2015
Forecasting for period starting 2015 to 2016
Forecasting for period starting 2016 to 2017
Forecasting for period starting 2017 to 2018
Forecasting for period starting 2018 to 2019
Forecasting for period starting 2019 to 2020
Forecasting for period starting 2020 to 2021
Forecasting for period starting 2021 to 2022
Forecasting for period starting 2022 to 2023
Forecasting for period starting 2023 to 2024


## Linear Classification

In [13]:
df_metrics_lm_clf = pd.DataFrame(columns=['start_date', 'end_date', 'model'])
df_results_lm_clf = pd.DataFrame(columns=['bond_cusip', 'date', 'model', 'pred', 'act'])

lm_clf = LogisticRegression(max_iter=1000, random_state=random_seed, n_jobs=-1)

for year_ in range(start_year, end_year + 1):

    print(f"Forecasting for period starting {year_ + train_year + val_year} to {year_ + train_year + val_year + test_year}")
    X_train, y_train, idx_train, X_val, y_val, idx_val, X_test, y_test, idx_test = data_split(data, x_cols, y_col_clf, idx_cols,
                                                                                     year_, train_year, val_year, test_year)
    X_train_full = pd.concat([X_train, X_val], ignore_index=True)
    y_train_full = pd.concat([y_train, y_val], ignore_index=True)

    lm_clf.fit(X_train_full, y_train_full)
    y_pred = lm_clf.predict(X_test)
    accuracy, precision, recall = calc_metrics(y_pred, y_test, task_type='classification')
    df = pd.DataFrame({'start_date': [year_ + train_year + val_year],
                       'end_date': [year_ + train_year + val_year + test_year],
                       'model': ['Logistic Regression'], 
                       'accuracy': [accuracy],
                       'precision': [precision],
                       'recall': [recall]})

    df_result = pd.DataFrame({'bond_cusip': idx_test['bond_cusip'],
                              'date': idx_test['date'],
                              'model': ['Logistic Regression'] * len(y_test),
                              'pred': y_pred,
                              'act': y_test})
    df_metrics_lm_clf = pd.concat([df_metrics_lm_clf, df], ignore_index=True)
    df_results_lm_clf = pd.concat([df_results_lm_clf, df_result], ignore_index=True)

Forecasting for period starting 2013 to 2014


/var/folders/hw/dfkspcjj1bv53cx3_42lwrtc0000gn/T/ipykernel_40006/3031172975.py:30: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_results_lm_clf = pd.concat([df_results_lm_clf, df_result], ignore_index=True)


Forecasting for period starting 2014 to 2015
Forecasting for period starting 2015 to 2016
Forecasting for period starting 2016 to 2017
Forecasting for period starting 2017 to 2018
Forecasting for period starting 2018 to 2019
Forecasting for period starting 2019 to 2020
Forecasting for period starting 2020 to 2021
Forecasting for period starting 2021 to 2022
Forecasting for period starting 2022 to 2023
Forecasting for period starting 2023 to 2024


## Elastic Net Classification

In [14]:
# Elastic Net Classification

df_metrics_en_clf = pd.DataFrame(columns=['start_date', 'end_date', 'model'])
df_results_en_clf = pd.DataFrame(columns=['bond_cusip', 'date', 'model', 'pred', 'act'])
hypoerparams_en_clf = {}
for year_ in range(start_year, end_year + 1):
    print(f"Forecasting for period starting {year_ + train_year + val_year} to {year_ + train_year + val_year + test_year}")
    X_train, y_train, idx_train, X_val, y_val, idx_val, X_test, y_test, idx_test = data_split(data, x_cols, y_col_clf, idx_cols,
                                                                                     year_, train_year, val_year, test_year)
    X_train_full = pd.concat([X_train, X_val], ignore_index=True)
    y_train_full = pd.concat([y_train, y_val], ignore_index=True)

    elastic_net_clf = LogisticRegressionCV(penalty='elasticnet', 
                                           solver='saga', 
                                           l1_ratios=l1_ratios,
                                           Cs=[1/alpha for alpha in alphas],
                                           cv=5,
                                           max_iter=1000,
                                           random_state=random_seed,
                                           n_jobs=-1)
    elastic_net_clf.fit(X_train_full, y_train_full)
    hypoerparams_en_clf[year_] = {'alpha': 1/elastic_net_clf.C_[0], 'l1_ratio': elastic_net_clf.l1_ratio_[0]}

    y_pred = elastic_net_clf.predict(X_test)
    accuracy, precision, recall = calc_metrics(y_pred, y_test, task_type='classification')
    df = pd.DataFrame({'start_date': [year_ + train_year + val_year],
                       'end_date': [year_ + train_year + val_year + test_year],
                       'model': ['Elastic Net Classifier'],
                       'accuracy': [accuracy],
                       'precision': [precision],
                       'recall': [recall]})
    df_result = pd.DataFrame({'bond_cusip': idx_test['bond_cusip'],
                              'date': idx_test['date'],
                              'model': ['Elastic Net Classifier'] * len(y_test),
                              'pred': y_pred,
                              'act': y_test})  
    df_metrics_en_clf = pd.concat([df_metrics_en_clf, df], ignore_index=True)
    df_results_en_clf = pd.concat([df_results_en_clf, df_result], ignore_index=True)



Forecasting for period starting 2013 to 2014


/opt/homebrew/Caskroom/miniconda/base/envs/ml/lib/python3.13/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/opt/homebrew/Caskroom/miniconda/base/envs/ml/lib/python3.13/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/var/folders/hw/dfkspcjj1bv53cx3_42lwrtc0000gn/T/ipykernel_40006/3152779335.py:38: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_results_en_clf = pd.concat([df_results_en_clf, df_result], ignore_index=True)


Forecasting for period starting 2014 to 2015


/opt/homebrew/Caskroom/miniconda/base/envs/ml/lib/python3.13/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/opt/homebrew/Caskroom/miniconda/base/envs/ml/lib/python3.13/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


Forecasting for period starting 2015 to 2016


...

/opt/homebrew/Caskroom/miniconda/base/envs/ml/lib/python3.13/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/opt/homebrew/Caskroom/miniconda/base/envs/ml/lib/python3.13/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


Forecasting for period starting 2022 to 2023


/opt/homebrew/Caskroom/miniconda/base/envs/ml/lib/python3.13/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/opt/homebrew/Caskroom/miniconda/base/envs/ml/lib/python3.13/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


Forecasting for period starting 2023 to 2024


/opt/homebrew/Caskroom/miniconda/base/envs/ml/lib/python3.13/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/opt/homebrew/Caskroom/miniconda/base/envs/ml/lib/python3.13/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


## Boosting Classification (LightGBM)

In [15]:


hypoerparams_lgmb_clf = {}
df_metrics_lgbm_clf = pd.DataFrame(columns=['start_date', 'end_date', 'model'])
df_results_lgbm_clf = pd.DataFrame(columns=['bond_cusip', 'date', 'model', 'pred', 'act'])

callbacks = [early_stopping(stopping_rounds=50)]

for year_ in range(start_year, end_year + 1):
    print(f"Tuning and Forecasting for period starting {year_ + train_year + val_year} to {year_ + train_year + val_year + test_year}")
    X_train, y_train, idx_train, X_val, y_val, idx_val, X_test, y_test, idx_test = data_split(data, x_cols, y_col_clf, idx_cols,
                                                                                        year_, train_year, val_year, test_year)
    y_train_full = pd.concat([y_train, y_val], ignore_index=True)
    best_accuracy_clf = 0
    # Tuning max_leaves and learning_rate
    for leaves in max_leaves:
        for speed in learning_speed:
            lgbm_clf = LGBMClassifier(max_leaves=leaves, n_estimators=num_tree, learning_rate=speed, random_state=random_seed, n_jobs=-1, verbose=-1)
            lgbm_clf.fit(X_train, y_train,
                         eval_set=[(X_val, y_val)],
                         callbacks=callbacks)
            y_val_pred = lgbm_clf.predict(X_val)
            accuracy, _, _ = calc_metrics(y_val_pred, y_val, task_type='classification')
            if accuracy > best_accuracy_clf:
                best_accuracy_clf = accuracy
                hypoerparams_lgmb_clf[year_] = {'num_leaves': leaves, 'learning_rate': speed}
    # Predict with best params
    lgbm_clf = LGBMClassifier(max_leaves=hypoerparams_lgmb_clf[year_]['num_leaves'],
                              n_estimators=num_tree,
                              learning_rate=hypoerparams_lgmb_clf[year_]['learning_rate'],
                              random_state=random_seed, n_jobs=-1, verbose=-1)
    lgbm_clf.fit(pd.concat([X_train, X_val], ignore_index=True),
                  pd.concat([y_train, y_val], ignore_index=True),
                  eval_set=[(X_val, y_val)],
                      callbacks=callbacks)
    y_pred = lgbm_clf.predict(X_test)
    accuracy, precision, recall = calc_metrics(y_pred, y_test, task_type='classification')
    df = pd.DataFrame({'start_date': [year_ + train_year + val_year],
                       'end_date': [year_ + train_year + val_year + test_year],
                       'model': ['LGBM Classifier'],
                       'accuracy': [accuracy],      
                       'precision': [precision],
                       'recall': [recall]})
    df_result = pd.DataFrame({'bond_cusip': idx_test['bond_cusip'],
                              'date': idx_test['date'],
                              'model': ['LGBM Classifier'] * len(y_test),
                              'pred': y_pred,
                              'act': y_test})  
    df_metrics_lgbm_clf = pd.concat([df_metrics_lgbm_clf, df], ignore_index=True)
    df_results_lgbm_clf = pd.concat([df_results_lgbm_clf, df_result], ignore_index=True)



Tuning and Forecasting for period starting 2013 to 2014
Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[456]	valid_0's multi_logloss: 0.690955
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[261]	valid_0's multi_logloss: 0.690751
Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[456]	valid_0's multi_logloss: 0.690955
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[261]	valid_0's multi_logloss: 0.690751
Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[456]	valid_0's multi_logloss: 0.690955
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[261]	valid_0's multi_logloss: 0.690751
Training until validation scores don't improve for 50 rounds
Did not meet early stopping.

/var/folders/hw/dfkspcjj1bv53cx3_42lwrtc0000gn/T/ipykernel_40006/2016743904.py:48: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_results_lgbm_clf = pd.concat([df_results_lgbm_clf, df_result], ignore_index=True)


Tuning and Forecasting for period starting 2014 to 2015
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[295]	valid_0's multi_logloss: 0.675803
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[99]	valid_0's multi_logloss: 0.682069
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[295]	valid_0's multi_logloss: 0.675803
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[99]	valid_0's multi_logloss: 0.682069
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[295]	valid_0's multi_logloss: 0.675803
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[99]	valid_0's multi_logloss: 0.682069
Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[500]	valid_0's multi_

In [16]:
from sklearn.ensemble import StackingClassifier

df_metrics_stack_clf = pd.DataFrame(columns=['start_date', 'end_date', 'model'])
df_results_stack_clf = pd.DataFrame(columns=['bond_cusip', 'date', 'model', 'pred', 'act'])

for year_ in range(start_year, end_year + 1):
    print(f"Forecasting for period starting {year_ + train_year + val_year} to {year_ + train_year + val_year + test_year}")
    X_train, y_train, idx_train, X_val, y_val, idx_val, X_test, y_test, idx_test = data_split(data, x_cols, y_col_clf, idx_cols,
                                                                                     year_, train_year, val_year, test_year)
    X_train_full = pd.concat([X_train, X_val], ignore_index=True)
    y_train_full = pd.concat([y_train, y_val], ignore_index=True)

    lm_clf = LogisticRegression(max_iter=1000, random_state=random_seed, n_jobs=-1)
    best_alpha_clf = hypoerparams_en_clf[year_]['alpha']
    best_l1_ratio_clf = hypoerparams_en_clf[year_]['l1_ratio']
    elastic_net_clf = LogisticRegression(penalty='elasticnet', 
                                        solver='saga', 
                                        l1_ratio=best_l1_ratio_clf, 
                                        C=1/best_alpha_clf, 
                                        max_iter=1000,
                                        random_state=random_seed, n_jobs=-1)
    best_params_clf = hypoerparams_lgmb_clf[year_]
    lgbm_clf = LGBMClassifier(max_leaves=best_params_clf['num_leaves'],
                              n_estimators=num_tree,
                              learning_rate=best_params_clf['learning_rate'],
                              random_state=random_seed, n_jobs=-1, verbose=-1)
    base_models_clf = [
        ('logistic_regression', lm_clf),
        ('elastic_net', elastic_net_clf),
        ('lgbm', lgbm_clf)
    ]
    meta_model_clf = LogisticRegression(max_iter=1000, random_state=random_seed, n_jobs=-1)
    stacked_model_clf = StackingClassifier(estimators=base_models_clf, final_estimator=meta_model_clf, n_jobs=-1)
    stacked_model_clf.fit(X_train_full, y_train_full)
    y_pred = stacked_model_clf.predict(X_test)
    accuracy, precision, recall = calc_metrics(y_pred, y_test, task_type='classification')
    df = pd.DataFrame({'start_date': [year_ + train_year + val_year],
                       'end_date': [year_ + train_year + val_year + test_year],
                       'model': ['Stacking Classifier'],
                       'accuracy': [accuracy],
                       'precision': [precision],
                       'recall': [recall]})
    df_result = pd.DataFrame({'bond_cusip': idx_test['bond_cusip'],
                              'date': idx_test['date'],
                              'model': ['Stacking Classifier'] * len(y_test),
                              'pred': y_pred,
                              'act': y_test})  
    df_metrics_stack_clf = pd.concat([df_metrics_stack_clf, df], ignore_index=True)
    df_results_stack_clf = pd.concat([df_results_stack_clf, df_result], ignore_index=True)  


Forecasting for period starting 2013 to 2014


/var/folders/hw/dfkspcjj1bv53cx3_42lwrtc0000gn/T/ipykernel_40006/1243034196.py:49: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_results_stack_clf = pd.concat([df_results_stack_clf, df_result], ignore_index=True)


Forecasting for period starting 2014 to 2015
Forecasting for period starting 2015 to 2016
Forecasting for period starting 2016 to 2017
Forecasting for period starting 2017 to 2018


/opt/homebrew/Caskroom/miniconda/base/envs/ml/lib/python3.13/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


Forecasting for period starting 2018 to 2019
Forecasting for period starting 2019 to 2020
Forecasting for period starting 2020 to 2021
Forecasting for period starting 2021 to 2022
Forecasting for period starting 2022 to 2023
Forecasting for period starting 2023 to 2024


/opt/homebrew/Caskroom/miniconda/base/envs/ml/lib/python3.13/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


In [17]:
df_results_clf = pd.concat([df_results_lm_clf, df_results_en_clf, df_results_lgbm_clf, df_results_stack_clf], ignore_index=True)
df_metrics_clf = pd.concat([df_metrics_lm_clf, df_metrics_en_clf, df_metrics_lgbm_clf, df_metrics_stack_clf], ignore_index=True)

In [18]:

df_results_reg = pd.concat([df_results_lm, df_results_en, df_results_lgbm, df_results_stack], ignore_index=True)
df_metrics_reg = pd.concat([df_metrics_lm, df_metrics_en, df_metrics_lgbm, df_metrics_stack], ignore_index=True)

In [19]:
ave_reg = df_metrics_reg.groupby('model')[['mse', 'mae', 'medae', 'r2']].mean().reset_index()
print(ave_reg)


               model       mse       mae     medae        r2
0         ElasticNet  0.000097  0.003107  0.001900 -0.019272
1      LGBMRegressor  0.000089  0.003069  0.001620 -0.109083
2   LinearRegression  0.000094  0.003226  0.002065  0.001753
3  StackingRegressor  0.000086  0.002959  0.001591 -0.008823


In [20]:

ave_clf = df_metrics_clf.groupby('model')[['accuracy', 'precision', 'recall']].mean().reset_index()
print(ave_clf)

                    model  accuracy  precision    recall
0  Elastic Net Classifier  0.687915   0.686969  0.687915
1         LGBM Classifier  0.713470   0.719949  0.713470
2     Logistic Regression  0.687858   0.687190  0.687858
3     Stacking Classifier  0.713615   0.720544  0.713615


In [21]:
from sklearn.metrics import confusion_matrix

for model_name in df_results_clf['model'].unique():
    df_model = df_results_clf[df_results_clf['model'] == model_name]
    cm = confusion_matrix(df_model['act'], df_model['pred'], labels=['up', 'down', 'neutral'])
    # Add labels
    cm_df = pd.DataFrame(cm, index=['up', 'down', 'neutral'], columns=['up', 'down', 'neutral'])
    print(f"Confusion Matrix for {model_name}:\n{cm_df}\n")

Confusion Matrix for Logistic Regression:
             up    down  neutral
up       155810  103344      200
down      52584  207780      286
neutral    4410    6969       79

Confusion Matrix for Elastic Net Classifier:
             up    down  neutral
up       155781  103385      188
down      52524  207861      265
neutral    4418    6975       65

Confusion Matrix for LGBM Classifier:
             up    down  neutral
up       176082   82742      530
down      62480  197594      576
neutral    2382    3005     6071

Confusion Matrix for Stacking Classifier:
             up    down  neutral
up       178684   80475      195
down      64062  196423      165
neutral    3217    4005     4236



In [22]:
print(len(data))

854769
